# 05 — Fetch Subscriptions

Pulls **every** subscription from MySQL (no `AccountCode` filter — this
loads the full set to be split across the two target accounts), then for
each one:

1. Decides which target account it belongs to (`resolve_target_account`,
   based on `Reference`).
2. Looks up its real address + radius username from Voyager
   (`get_voyager_address`) — `SupplierServiceID` -> `GET .../fibre/v1/circuits/{id}`
   (gives `radiusUsers[0]` + `locationId`) -> `GET .../address-search/v3/addresses/id/{locationId}`
   (gives the actual street address, city, postcode, region). Region name
   is mapped to a 3-char ISO code via `NZ_Regions.xlsx`.

This is a data-prep step that calls out to Voyager (not OneBill) — no
OneBill API calls happen here. Output feeds both `06_Create_Addresses.ipynb`
and `07_Create_Subscription_Orders.ipynb` (the latter now uses the resolved
`radius_user` instead of `SubscriptionLabel`).

> **TODO**: confirm the `Reference` column name
> (`SUBSCRIPTION_REFERENCE_COLUMN` in `onebill_common.py`, currently
> `"Reference"`).

## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from sqlalchemy import create_engine
from concurrent.futures import ThreadPoolExecutor

logger = get_logger("fetch_subscriptions")

# Use this to limit rows while testing. Set to None once ready for a full run.
TEST_ROW_LIMIT = 5


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 23
python-dotenv could not parse statement starting at line 29


## 2. Pull every subscription from MySQL

In [7]:
assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
engine = create_engine(BI_DATASTORE_URL)

SUBSCRIPTION_QUERY = '''
SELECT 
    *
FROM bi_datastore.billing_subscription
WHERE _DataSource = 'vBill'
AND AccountCode = '99965692'
ORDER BY RAND();
'''.strip()

df_subscriptions = pd.read_sql(SUBSCRIPTION_QUERY, con=engine)
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from MySQL")

if TEST_ROW_LIMIT is not None:
    df_subscriptions = df_subscriptions.head(TEST_ROW_LIMIT)  # Testing limiter — remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active — trimmed to {len(df_subscriptions):,} rows")

df_subscriptions.head()


2026-07-22 05:38:51,039 [INFO] Loaded 323 subscriptions from MySQL
2026-07-22 05:38:51,041 [INFO] TEST_ROW_LIMIT active — trimmed to 5 rows


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,CircuitType,Server,CustomerSuppliedReference,_notforreports_VoyagerOrderHistory,_notforreports_LegacyServiceDescription,NextPlanCode,NextPlanStartDate,NextQuantity,NextCustomPrice,SalesAgentCode
0,4335503110,2026-07-03 20:37:16,264492,vBill,99965692,Broadband - Fibre,V113091029,g03.18bath@williamsinternet.com,2026-05-20,None,...,UFB 100/20/2.5/2.5,None,Managed by Williams Limited,CPP 1007194,None,None,None,None,None,None
1,4167998688,2026-07-03 20:36:54,260847,vBill,99965692,Broadband - Fibre,V113062392,3matataway@williamsinternet.com,2025-12-01,None,...,UFB 100/20/2.5/2.5,None,WC CHCH T9,CPP 2006218,None,None,None,None,None,None
2,4277016830,2026-07-03 20:22:20,263114,vBill,99965692,Broadband - Fibre,V113080527,10.346@williamsinternet.com,2026-03-23,None,...,UFB 100/20/2.5/2.5,None,Managed by Williams Limited,CPP 1006990,None,None,None,None,None,None
3,4177633771,2026-07-03 20:37:01,261154,vBill,99965692,Broadband - Fibre,V113065130,V113065130_0@williamsinternet.com,2025-12-12,2026-03-25,...,UFB 100/20/2.5/2.5,None,WC CHCH T9,2026-03-24 RQ[CPP 1007012] ; CPP 1006657,None,None,None,None,None,None
4,4224347745,2026-07-03 20:37:20,261945,vBill,99965692,Broadband - Fibre,V113071302,11.7ariki@williamsinternet.com,2026-01-29,None,...,UFB 500/100/2.5/2.5,None,WC CHCH T9,CPP 3504432,None,None,None,None,None,None


## 3. Resolve target account for every subscription

`TargetAccountNumber` is now primarily the subscription's **own** account —
matched by `AccountCode` against `04_Create_Accounts.ipynb`'s real results
(`load_account_code_batch_map`), which is the actual OneBill `accountNumber`
(`AccountCode` + `.` + `BATCH_NUMBER`) for that account. This is what the
account is really called inside OneBill, so subscriptions land back on their
own account instead of one of the two shared Williams buckets.

If a subscription's own account has no successful (`created`/`exists`) row
in `account_results` — e.g. it wasn't migrated, or failed — it falls back to
the old two-bucket routing (`resolve_target_account`, based on `Reference`)
so the row can still be processed rather than being dropped outright.

Run `04_Create_Accounts.ipynb` before this notebook if you haven't already.

In [8]:
own_account_map = load_account_code_batch_map()  # {AccountCode: AccountCode_Batch}, every account in 04's results
real_account_numbers = load_real_target_account_numbers()  # {"managed_by_williams": "...", "williams_corporation": "..."} — fallback only

missing_keys = set(TARGET_ACCOUNTS) - set(real_account_numbers)
if missing_keys:
    logger.warning(
        f"No successful account_results row found for: {missing_keys} — "
        f"falling back to the TARGET_ACCOUNTS placeholder for those. "
        f"Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run."
    )

logger.info(f"{len(own_account_map):,} accounts available from 04_Create_Accounts.ipynb (own-account routing)")


def _bucket_account_key_and_number(reference):
    """Fallback routing for subscriptions whose own account isn't in own_account_map."""
    account = resolve_target_account(reference)
    account_key = next(k for k, v in TARGET_ACCOUNTS.items() if v is account)
    return pd.Series({
        "TargetAccountKey":    account_key,
        "TargetAccountNumber": real_account_numbers.get(account_key, account["account_number"]),
    })


reference_col = SUBSCRIPTION_REFERENCE_COLUMN if SUBSCRIPTION_REFERENCE_COLUMN in df_subscriptions.columns else None
if reference_col is None:
    logger.warning(
        f"Column '{SUBSCRIPTION_REFERENCE_COLUMN}' not found in df_subscriptions — "
        f"any subscription that needs the bucket fallback will default to 'Williams Corporation'. "
        f"Available columns: {list(df_subscriptions.columns)}"
    )
    df_subscriptions["CustomerSuppliedReference"] = None
else:
    df_subscriptions["CustomerSuppliedReference"] = df_subscriptions[reference_col]

df_subscriptions["AccountCode"] = df_subscriptions["AccountCode"].astype(str)
df_subscriptions["TargetAccountNumber"] = df_subscriptions["AccountCode"].map(own_account_map)
df_subscriptions["TargetAccountKey"] = df_subscriptions["TargetAccountNumber"].notna().map({True: "own_account", False: None})

missing_own_account = df_subscriptions["TargetAccountNumber"].isna()
if missing_own_account.any():
    logger.warning(
        f"{missing_own_account.sum():,} subscriptions have no matching created/existing account in "
        f"04_Create_Accounts.ipynb's results — falling back to the Williams bucket accounts for these."
    )
    bucket_fallback = df_subscriptions.loc[missing_own_account, "CustomerSuppliedReference"].apply(_bucket_account_key_and_number)
    df_subscriptions.loc[missing_own_account, "TargetAccountKey"]    = bucket_fallback["TargetAccountKey"]
    df_subscriptions.loc[missing_own_account, "TargetAccountNumber"] = bucket_fallback["TargetAccountNumber"]

logger.info(df_subscriptions["TargetAccountKey"].value_counts(dropna=False).to_string())
df_subscriptions[["SubscriptionUSN", "AccountCode", "TargetAccountKey", "TargetAccountNumber"]].head(20)


2026-07-22 05:38:55,508 [WARNING] No successful account_results row found for: {'williams_corporation'} — falling back to the TARGET_ACCOUNTS placeholder for those. Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run.
2026-07-22 05:38:55,510 [INFO] 2 accounts available from 04_Create_Accounts.ipynb (own-account routing)
2026-07-22 05:38:55,516 [INFO] TargetAccountKey
own_account    5


,SubscriptionUSN,AccountCode,TargetAccountKey,TargetAccountNumber
0,V113091029,99965692,own_account,99965692_10001
1,V113062392,99965692,own_account,99965692_10001
2,V113080527,99965692,own_account,99965692_10001
3,V113065130,99965692,own_account,99965692_10001
4,V113071302,99965692,own_account,99965692_10001


## 4. Look up each subscription's real address + radius username (Voyager)

Two API calls per subscription, keyed on `SupplierServiceID`:
`fetch_voyager_circuit` -> `fetch_voyager_address`, wrapped by
`get_voyager_address`. Run in parallel (it's now real network calls, not
pure string parsing) via the same `ThreadPoolExecutor` pattern used in
`06_Create_Addresses.ipynb`.

Subscriptions with no `SupplierServiceID`, or where either Voyager call
fails, come back with `parsed_ok = False` — flagged the same way unparsed
labels used to be, and skipped by `06_Create_Addresses.ipynb`.

In [9]:
if VOYAGER_CCP_KEY is None or VOYAGER_PARTNER_ID is None:
    logger.warning("VOYAGER_CCP_KEY / VOYAGER_PARTNER_ID not set — every Voyager lookup below will fail. Set them in .env.")

blank_supplier_ids = df_subscriptions["SupplierServiceID"].isna() | (df_subscriptions["SupplierServiceID"].astype(str).str.strip() == "")
if blank_supplier_ids.all():
    logger.warning(
        "SupplierServiceID is blank for EVERY subscription in this batch — the Voyager circuits lookup "
        "can't run at all without it, so every ParsedAddress_* field below will be None. Check the MySQL "
        "source data / SUBSCRIPTION_QUERY in step 2 before re-running."
    )
elif blank_supplier_ids.any():
    logger.warning(f"{blank_supplier_ids.sum():,} / {len(df_subscriptions):,} subscriptions have a blank SupplierServiceID")

voyager_session = new_voyager_session(max_workers=MAX_WORKERS)


def _lookup_row(supplier_service_id):
    return get_voyager_address(voyager_session, supplier_service_id)


logger.info(f"Looking up Voyager address for {len(df_subscriptions):,} subscriptions with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    voyager_results = list(executor.map(_lookup_row, df_subscriptions["SupplierServiceID"]))

address_parts = pd.DataFrame(voyager_results, index=df_subscriptions.index)
address_parts = address_parts.add_prefix("ParsedAddress_")
df_subscriptions = pd.concat([df_subscriptions, address_parts], axis=1)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    logger.warning(f"{len(unparsed):,} subscriptions could not be resolved to a Voyager address")
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info("Error breakdown:\n" + error_summary.to_string(index=False))

df_subscriptions[[
    "SubscriptionLabel", "SupplierServiceID",
    "ParsedAddress_addLine1", "ParsedAddress_addLine2", "ParsedAddress_city",
    "ParsedAddress_postcode", "ParsedAddress_region_iso", "ParsedAddress_region_code_raw", "ParsedAddress_radius_user",
    "ParsedAddress_parsed_ok", "ParsedAddress_error",
]].head(20)


2026-07-22 05:38:56,370 [INFO] Looking up Voyager address for 5 subscriptions with 10 workers...
2026-07-22 05:39:04,418 [WARNING] 1 subscriptions could not be resolved to a Voyager address
2026-07-22 05:39:04,421 [INFO] Error breakdown:
                                                                                                               error  count
circuits lookup failed: 404 Client Error: Not Found for url: https://api.voyager.nz/fibre/v1/circuits/ENVOYB02620633      1


,SubscriptionLabel,SupplierServiceID,ParsedAddress_addLine1,ParsedAddress_addLine2,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_parsed_ok,ParsedAddress_error
0,g03.18bath@williamsinternet.com,ENVOYB02648625,G03/18 Bath Street,Christchurch Central,Christchurch,8011,CAN,CAN,g03.18bath@williamsinternet.com,True,None
1,3matataway@williamsinternet.com,UFF000007009635,3 Matata Way,Tauranga,Tauranga,3110,BOP,BOP,3matataway@williamsinternet.com,True,None
2,10.346@williamsinternet.com,ENVOYB02636755,10/346 Cashel Street,Christchurch Central,Christchurch,8011,CAN,CAN,10.346@williamsinternet.com,True,None
3,V113065130_0@williamsinternet.com,ENVOYB02620633,None,None,None,None,None,None,None,False,circuits lookup failed: 404 Client Error: Not ...
4,11.7ariki@williamsinternet.com,1643230569,11/7 Ariki Street,Boulcott,Lower Hutt,5010,WGN,WGN,11.7ariki@williamsinternet.com,True,None


In [10]:
unparsed[['SubscriptionUSN', 'SupplierServiceID', 'ParsedAddress_error']] if not unparsed.empty else 'No unresolved addresses \N{WHITE HEAVY CHECK MARK}'

,SubscriptionUSN,SupplierServiceID,ParsedAddress_error
3,V113065130,ENVOYB02620633,circuits lookup failed: 404 Client Error: Not ...


## 5. Save

In [11]:
save_df("subscriptions_resolved", df_subscriptions)


Saved 5 rows -> migration_data\05_subscriptions_resolved.csv
